# 주식옵션 실시간 호가 수집기 (MM 델타 헷징 연구용)
키움 Open API를 사용하여 주식옵션 호가/시세(그릭스, IV 포함) 및 기초자산 체결 데이터를 CSV로 저장

## 1. 설정

In [ ]:
# === 사용자 설정 ===
OPTION_CODE = "B1164089"  # 삼성전자 2026년 4월물 콜옵션 (ATM 185,000원)
STOCK_CODE = "005930"     # 기초자산 종목코드 (삼성전자 주식)
SCREEN_NO = "1000"        # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop, QTimer

## 3. CSV 저장 설정 및 FID 매핑

In [ ]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"option_{OPTION_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의 (옵션 MM 델타 헷징 연구용 핵심 컬럼)
CSV_COLUMNS = [
    # 기본 정보
    "timestamp", "data_type",

    # 옵션 시세 (옵션시세 Real Type)
    "option_code", "option_time", "option_price", "option_volume",
    "option_theoretical",       # 이론가
    "option_iv",                # 내재변동성 (Implied Volatility)
    "option_delta",             # 델타
    "option_gamma",             # 감마
    "option_theta",             # 세타
    "option_vega",              # 베가
    "option_rho",               # 로
    "option_open_interest",     # 미결제약정

    # 옵션 호가 (옵션호가잔량 Real Type)
    "option_ask_1", "option_ask_qty_1",
    "option_ask_2", "option_ask_qty_2",
    "option_ask_3", "option_ask_qty_3",
    "option_ask_4", "option_ask_qty_4",
    "option_ask_5", "option_ask_qty_5",
    "option_bid_1", "option_bid_qty_1",
    "option_bid_2", "option_bid_qty_2",
    "option_bid_3", "option_bid_qty_3",
    "option_bid_4", "option_bid_qty_4",
    "option_bid_5", "option_bid_qty_5",
    "option_ask_total", "option_bid_total",

    # 기초자산 (주식체결 Real Type)
    "stock_code", "stock_time", "stock_price"
]

# FID 매핑 - 옵션시세
FID_OPTION_TRADE = {
    "option_time": 20,              # 체결시간
    "option_price": 10,             # 현재가
    "option_volume": 15,            # 거래량
    "option_theoretical": 182,      # 이론가
    "option_iv": 190,               # 내재변동성
    "option_delta": 191,            # 델타
    "option_gamma": 192,            # 감마
    "option_theta": 193,            # 세타
    "option_vega": 194,             # 베가
    "option_rho": 196,              # 로
    "option_open_interest": 195,    # 미결제약정
}

# FID 매핑 - 옵션호가잔량
FID_OPTION_QUOTE = {
    # 매도호가
    "option_ask_1": 41, "option_ask_2": 42, "option_ask_3": 43, "option_ask_4": 44, "option_ask_5": 45,
    # 매도잔량
    "option_ask_qty_1": 61, "option_ask_qty_2": 62, "option_ask_qty_3": 63, "option_ask_qty_4": 64, "option_ask_qty_5": 65,
    # 매수호가
    "option_bid_1": 51, "option_bid_2": 52, "option_bid_3": 53, "option_bid_4": 54, "option_bid_5": 55,
    # 매수잔량
    "option_bid_qty_1": 71, "option_bid_qty_2": 72, "option_bid_qty_3": 73, "option_bid_qty_4": 74, "option_bid_qty_5": 75,
    # 총잔량
    "option_ask_total": 121,
    "option_bid_total": 125,
}

# FID 매핑 - 주식체결 (기초자산 - 델타 헷징 기준가)
FID_STOCK_TRADE = {
    "stock_time": 20,
    "stock_price": 10,
}

# 전체 FID 통합 (실시간 등록용)
ALL_FIDS = set(FID_OPTION_TRADE.values()) | set(FID_OPTION_QUOTE.values()) | set(FID_STOCK_TRADE.values())

## 4. KiwoomOptionCollector 클래스 정의

In [ ]:
class KiwoomOptionCollector:
    def __init__(self, option_code, stock_code, screen_no, csv_path, interval_ms=1000):
        self.option_code = option_code
        self.stock_code = stock_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        self.interval_ms = interval_ms  # 저장 간격 (밀리초)

        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")

        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)

        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0

        # 마지막 수신 데이터 캐시
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["option_code"] = option_code
        self.last_data["stock_code"] = stock_code

        # 데이터 수신 여부 플래그
        self.data_received = False

        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()

        # 1초 타이머 설정
        self.timer = QTimer()
        self.timer.timeout.connect(self.on_timer)

        # CSV 헤더 작성
        self._init_csv()

    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")

    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()

    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True

            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")

            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False

        self.login_event_loop.exit()

    def set_real_reg(self):
        """실시간 등록 (옵션 + 기초자산)"""
        fid_list = ";".join([str(fid) for fid in ALL_FIDS])

        # 옵션 종목 등록
        result1 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.option_code, fid_list, "0"
        )
        print(f"옵션 실시간 등록 ({self.option_code}): {result1}")

        # 기초자산(주식) 종목 추가 등록 (opt_type="1"로 추가)
        result2 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.stock_code, fid_list, "1"
        )
        print(f"주식 실시간 등록 ({self.stock_code}): {result2}")

        return result1, result2

    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value

    def _clean_value(self, value):
        """값 정리 (부호 제거 - 가격류만)"""
        if value and (value.startswith("+") or value.startswith("-")):
            if value[1:].replace(".", "").isdigit():
                return value[1:]
        return value

    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 → 캐시 업데이트만 (저장은 타이머에서)"""

        # [디버깅] 새로운 real_type이 들어오면 출력
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' (code: {code}) <<<")
            
            # 디버깅: 주식옵션시세일 때 모든 FID 값 출력
            if real_type == "주식옵션시세":
                print("[DEBUG] 주식옵션시세 FID 값 확인:")
                for fid in [10, 11, 12, 13, 15, 16, 17, 18, 20, 27, 28, 41, 51, 61, 71, 
                           121, 125, 181, 182, 183, 184, 185, 186, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220]:
                    val = self.get_real_data(code, fid)
                    if val:
                        print(f"  FID {fid}: '{val}'")

        # 처리할 실시간 타입 (주식옵션용)
        valid_types = (
            "주식옵션시세",       # 주식옵션 체결/시세
            "주식옵션호가잔량",   # 주식옵션 호가
            "주식체결",           # 기초자산 체결
        )

        if real_type not in valid_types:
            return

        # 타입별로 FID 조회 및 캐시 업데이트
        if real_type == "주식옵션시세" and code == self.option_code:
            self.last_data["data_type"] = "option_trade"
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        elif real_type == "주식옵션호가잔량" and code == self.option_code:
            self.last_data["data_type"] = "option_quote"
            for key, fid in FID_OPTION_QUOTE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        elif real_type == "주식체결" and code == self.stock_code:
            self.last_data["data_type"] = "stock_trade"
            for key, fid in FID_STOCK_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

    def on_timer(self):
        """1초마다 현재 캐시 상태를 CSV에 저장"""
        # 최소 한 번이라도 데이터를 받은 경우에만 저장
        if not self.data_received:
            return

        # 타임스탬프 업데이트
        self.last_data["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.last_data["data_type"] = "snapshot"  # 1초 스냅샷임을 표시

        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")

        # 콘솔 출력
        print(f"[{self.data_count:4d}] {self.last_data['timestamp']} | "
              f"옵션: {self.last_data.get('option_price', ''):>8} | "
              f"IV: {self.last_data.get('option_iv', ''):>6} | "
              f"Δ: {self.last_data.get('option_delta', ''):>7} | "
              f"주식: {self.last_data.get('stock_price', ''):>10}")

    def run(self):
        """메인 실행"""
        # 종목코드 검증
        if not self.option_code:
            print("[오류] OPTION_CODE가 설정되지 않았습니다.")
            print("상단 설정 셀에서 옵션 종목코드를 입력해주세요.")
            return

        self.login()

        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return

        print("\n" + "=" * 60)
        print(f"주식옵션 실시간 호가 수집 시작 (델타 헷징 연구용)")
        print(f"  - 옵션: {self.option_code}")
        print(f"  - 기초자산: {self.stock_code}")
        print(f"  - 저장 간격: {self.interval_ms}ms")
        print("=" * 60)

        self.set_real_reg()

        # 타이머 시작
        self.timer.start(self.interval_ms)

        print(f"\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print(f"저장 방식: {self.interval_ms/1000:.1f}초마다 스냅샷 저장\n")

        self.app.exec_()

    def stop(self):
        """수집 중지"""
        self.timer.stop()
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomOptionCollector(
    option_code=OPTION_CODE,
    stock_code=STOCK_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH,
    interval_ms=1000  # 1초마다 스냅샷 저장 (변경 가능: 500ms, 2000ms 등)
)

# 실행 (중지하려면 Kernel > Interrupt 또는 Ctrl+C)
collector.run()

In [ ]:
import pandas as pd

# 데이터 로드
df = pd.read_csv(r"c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\data\option_B1164089_20260313_1202.csv", encoding="utf-8-sig")

print(f"총 행 수: {len(df)}")
print(f"수집 시간 범위: {df['timestamp'].iloc[0]} ~ {df['timestamp'].iloc[-1]}")
print(f"\n=== 컬럼별 데이터 유무 ===")
for col in df.columns:
    non_null = df[col].notna().sum()
    non_empty = (df[col] != "").sum() if df[col].dtype == object else non_null
    print(f"{col}: {non_empty}/{len(df)} ({non_empty/len(df)*100:.1f}%)")